In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

PROJECT = Path("/content/drive/MyDrive/mitosis_detection_project")

print("Project exists:", PROJECT.exists())
print("Project path:", PROJECT)

paths_to_check = [
    "checkpoints/augmentation_experiment_v1_best.pt",
    "runs/detect/baseline_yolov8n/weights/best.pt",
    "runs/detect/augmentation_experiment_v1/weights/best.pt",
    "runs/detect/augmentation_experiment_v1/weights/last.pt",
    "runs/detect/baseline_yolov8n/results.csv",
    "runs/detect/augmentation_experiment_v1/results.csv",
    "configs/mitosis_data.yaml",
    "results/metrics/test_evaluation/ultralytics_metrics.json",
    "results/metrics/test_evaluation/custom_metrics.json",
]

print("\n--- IMPORTANT FILES ---")
for relative_path in paths_to_check:
    path = PROJECT / relative_path
    if path.exists():
        print(f"FOUND  | {relative_path} | {path.stat().st_size / 1e6:.2f} MB")
    else:
        print(f"MISSING | {relative_path}")

print("\n--- CHECKPOINTS DIRECTORY ---")
checkpoint_dir = PROJECT / "checkpoints"
if checkpoint_dir.exists():
    for path in checkpoint_dir.iterdir():
        print(path.name, f"({path.stat().st_size / 1e6:.2f} MB)")
else:
    print("Checkpoint directory not found")

Mounted at /content/drive
Project exists: True
Project path: /content/drive/MyDrive/mitosis_detection_project

--- IMPORTANT FILES ---
FOUND  | checkpoints/augmentation_experiment_v1_best.pt | 6.50 MB
FOUND  | runs/detect/baseline_yolov8n/weights/best.pt | 6.50 MB
FOUND  | runs/detect/augmentation_experiment_v1/weights/best.pt | 6.50 MB
FOUND  | runs/detect/augmentation_experiment_v1/weights/last.pt | 6.50 MB
FOUND  | runs/detect/baseline_yolov8n/results.csv | 0.00 MB
FOUND  | runs/detect/augmentation_experiment_v1/results.csv | 0.00 MB
FOUND  | configs/mitosis_data.yaml | 0.00 MB
FOUND  | results/metrics/test_evaluation/ultralytics_metrics.json | 0.00 MB
FOUND  | results/metrics/test_evaluation/custom_metrics.json | 0.00 MB

--- CHECKPOINTS DIRECTORY ---
augmentation_experiment_v1_best.pt (6.50 MB)


In [ ]:
%pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 7.7 MB/s eta 0:00:00


In [ ]:
import torch
import ultralytics

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics: 8.4.183
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [ ]:
from pathlib import Path
import pandas as pd
from ultralytics import YOLO

PROJECT = Path("/content/drive/MyDrive/mitosis_detection_project")
DATA = str(PROJECT / "configs/mitosis_data.yaml")

models = {
    "baseline": PROJECT / "runs/detect/baseline_yolov8n/weights/best.pt",
    "augmentation_v1": PROJECT / "runs/detect/augmentation_experiment_v1/weights/best.pt",
}

rows = []

for name, checkpoint in models.items():
    print(f"\nEvaluating {name}...")
    model = YOLO(str(checkpoint))
    metrics = model.val(
        data=DATA,
        split="val",
        imgsz=640,
        batch=8,
        conf=0.001,
        iou=0.7,
        plots=False,
        verbose=False,
    )

    rows.append({
        "model": name,
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "mAP50": float(metrics.box.map50),
        "mAP50_95": float(metrics.box.map),
    })

comparison = pd.DataFrame(rows)
out = PROJECT / "results/metrics/final_model_comparison.csv"
comparison.to_csv(out, index=False)

display(comparison)
print("Saved comparison to:", out)


Evaluating baseline...
Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.6±0.2 ms, read: 0.2±0.0 MB/s, size: 89.5 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/mitosis_detection_project/data/mitosis-gjs3g/valid/labels... 61 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 61/61 1.4it/s 44.3s
val: New cache created: /content/drive/MyDrive/mitosis_detection_project/data/mitosis-gjs3g/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 7.5it/s 1.1s
                   all         61         91      0.791      0.835       0.88      0.496
Speed: 0.9ms preprocess, 8.4ms inference, 0.0ms loss, 1.2ms postprocess per imag

,model,precision,recall,mAP50,mAP50_95
0,baseline,0.790677,0.835165,0.879681,0.495824
1,augmentation_v1,0.881298,0.897487,0.919494,0.573943


Saved comparison to: /content/drive/MyDrive/mitosis_detection_project/results/metrics/final_model_comparison.csv


In [ ]:
from pathlib import Path
import shutil
import json

PROJECT = Path("/content/drive/MyDrive/mitosis_detection_project")

source = PROJECT / "runs/detect/augmentation_experiment_v1/weights/best.pt"
destination = PROJECT / "checkpoints/final_mitosis_detector.pt"

assert source.exists(), f"Checkpoint missing: {source}"

shutil.copy2(source, destination)

summary = {
    "final_model": "YOLOv8n with conservative augmentation",
    "checkpoint": str(destination),
    "selection_basis": "Higher validation mAP50-95 than baseline",
    "validation_split": "valid",
    "imgsz": 640,
    "validation_metrics": {
        "precision": 0.881298,
        "recall": 0.897487,
        "mAP50": 0.919494,
        "mAP50_95": 0.573943
    },
    "test_set_note": "Previously evaluated; not an untouched test set"
}

summary_path = PROJECT / "results/metrics/final_model_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

print("Final checkpoint:", destination)
print("Checkpoint size:", round(destination.stat().st_size / 1e6, 2), "MB")
print("Summary saved:", summary_path)

Final checkpoint: /content/drive/MyDrive/mitosis_detection_project/checkpoints/final_mitosis_detector.pt
Checkpoint size: 6.5 MB
Summary saved: /content/drive/MyDrive/mitosis_detection_project/results/metrics/final_model_summary.json


In [ ]:
from pathlib import Path
import time
import numpy as np
import torch
from ultralytics import YOLO
import cv2

PROJECT = Path("/content/drive/MyDrive/mitosis_detection_project")
MODEL_PATH = PROJECT / "checkpoints/final_mitosis_detector.pt"
IMAGE_DIR = PROJECT / "data/mitosis-gjs3g/valid/images"

model = YOLO(str(MODEL_PATH))
image_paths = sorted(
    p for p in IMAGE_DIR.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
)

assert image_paths, f"No images found in {IMAGE_DIR}"

# Preload images so disk I/O does not dominate the timing.
images = []
for path in image_paths:
    image = cv2.imread(str(path))
    if image is not None:
        images.append((path.name, image))

assert images, "Could not read validation images"

device = 0 if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if device == 0 else "CPU")
print("Images:", len(images))

# Warm up the model.
for _, image in images[:min(5, len(images))]:
    model.predict(image, imgsz=640, conf=0.35, device=device, verbose=False)
if torch.cuda.is_available():
    torch.cuda.synchronize()

# Measure single-image end-to-end prediction latency.
latencies = []
detections = 0

for _, image in images:
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()

    result = model.predict(
        image, imgsz=640, conf=0.35, iou=0.7,
        device=device, verbose=False
    )[0]

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    latencies.append((time.perf_counter() - start) * 1000)
    detections += len(result.boxes)

latencies = np.asarray(latencies)
total_seconds = latencies.sum() / 1000

benchmark = {
    "checkpoint_size_MB": round(MODEL_PATH.stat().st_size / 1e6, 2),
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "images_benchmarked": len(images),
    "mean_latency_ms": round(float(latencies.mean()), 2),
    "median_latency_ms": round(float(np.median(latencies)), 2),
    "p95_latency_ms": round(float(np.percentile(latencies, 95)), 2),
    "throughput_images_per_second": round(len(images) / total_seconds, 2),
    "confidence_threshold": 0.35,
    "nms_iou": 0.7,
}

for key, value in benchmark.items():
    print(f"{key}: {value}")

import json
out = PROJECT / "results/metrics/inference_benchmark.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(benchmark, indent=2))
print("Saved:", out)

Device: Tesla T4
Images: 61
checkpoint_size_MB: 6.5
device: Tesla T4
images_benchmarked: 61
mean_latency_ms: 17.8
median_latency_ms: 14.93
p95_latency_ms: 29.98
throughput_images_per_second: 56.19
confidence_threshold: 0.35
nms_iou: 0.7
Saved: /content/drive/MyDrive/mitosis_detection_project/results/metrics/inference_benchmark.json


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

drive_root = Path('/content/drive/MyDrive')

for f in drive_root.glob('*.ipynb'):
    print(f.name)

Mounted at /content/drive
